# Contract Clause Classification

Compares a zero-shot LLM with a fine-tuned transformer on CUAD contracts.  Both decide, for each test contract and clause type, whether the clause appears anywhere in the contract, and both are scored against CUAD's answer spans.

The zero-shot LLM reads each contract in chunks and stops at the first YES.  The fine-tuned model scores every clause type at once over overlapping 512-token windows.  Latency and cost are measured per contract.

This notebook calls the same code as `compare_classifiers.py`, in `utils/evaluation.py`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from config import config
from utils.data_loader import load_cuad_dataset, get_clause_distribution
from utils.llm_client import LLMClient
from utils.evaluation import evaluate_zero_shot, evaluate_fine_tuned
from utils.metrics import aggregate_metrics

## Configuration

In [ ]:
CONFIG = {
    'clause_types': list(config.data.clause_types),
    'max_test_contracts': None,  # None = the whole test split
    'train': True,               # False = load models/fine_tuned instead
}
CONFIG

## Load data

In [ ]:
test_contracts = load_cuad_dataset(split='test', max_samples=CONFIG['max_test_contracts'])
print(f'{len(test_contracts)} test contracts')
get_clause_distribution(test_contracts)

## Zero-shot LLM

In [ ]:
zero_shot = evaluate_zero_shot(LLMClient(), test_contracts, CONFIG['clause_types'])
print(f'{zero_shot.calls} LLM calls')
{ct: n for ct, n in zero_shot.failures.items() if n}  # decisions left out after failed calls

## Fine-tuned model

In [ ]:
import os
from utils.classifier import FineTunedClassifier

model_path = os.path.join(config.paths.models_dir, 'fine_tuned')
if CONFIG['train']:
    fine_tuned = FineTunedClassifier(CONFIG['clause_types'])
    training_result = fine_tuned.train(
        load_cuad_dataset(split='train'),
        load_cuad_dataset(split='validation'),
    )
    print(f'Trained in {training_result.training_time_seconds:.0f} s on {training_result.train_windows} windows')
else:
    fine_tuned = FineTunedClassifier(model_name=model_path)
    fine_tuned.load(model_path)

In [ ]:
fine_tuned_result = evaluate_fine_tuned(
    fine_tuned, test_contracts, CONFIG['clause_types'],
    cost_per_hour=config.training.cost_per_hour,
)

## Comparison

In [ ]:
arms = {'Zero-Shot LLM': zero_shot, 'Fine-Tuned': fine_tuned_result}
rows = []
for ct in CONFIG['clause_types']:
    for label, arm in arms.items():
        m = arm.metrics.get(ct)
        if m:
            rows.append({'clause_type': ct, 'method': label, 'precision': m.precision,
                         'recall': m.recall, 'f1': m.f1, 'accuracy': m.accuracy})
per_clause = pd.DataFrame(rows)
per_clause.pivot(index='clause_type', columns='method', values='f1')

In [ ]:
pd.DataFrame({
    label: {
        'contracts': arm.stats.documents,
        'avg latency ms': round(arm.stats.avg_latency_ms),
        'min latency ms': round(arm.stats.min_latency_ms),
        'max latency ms': round(arm.stats.max_latency_ms),
        'cost per contract USD': arm.stats.avg_cost_usd,  # None = not priced
        'mean F1': aggregate_metrics(arm.metrics).get('avg_f1'),
    }
    for label, arm in arms.items()
})

## Plot

In [ ]:
ax = per_clause.pivot(index='clause_type', columns='method', values='f1').plot.bar(figsize=(12, 5))
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
plt.tight_layout()

## Save

In [ ]:
out = os.path.join(config.paths.outputs_dir, 'notebook_metrics.csv')
per_clause.to_csv(out, index=False)
out